In [2]:
import requests
import json
from bs4 import BeautifulSoup

URL = "https://vbpl.vn/van-ban/trung-uong"

#? Các header này được lấy từ request POST trong DevTools.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (hoc-crawl-cho-mon-hoc)",
    "Content-Type": "text/plain;charset=UTF-8",  #? Giá trị lấy từ DevTools.
    "Next-Action": "c529d164f28418e5898a834422629e64c6816af1",
    "Accept": "text/x-component",                #? Giá trị lấy từ DevTools.
    "Origin": "https://vbpl.vn",
    "Referer": URL,
}

#@ Tạo JSON body tìm kiếm cho trang và kích thước trang được yêu cầu.
def build_body(page_number, page_size=10):
    #? "$undefined" là chuỗi literal trong payload đã quan sát.
    payload = {
        "administrativeUnit": "$undefined",
        "agencyIds": "$undefined",
        "agencyLevel": "TRUNG_UONG",
        "docNum": "$undefined",
        "docType": "$undefined",
        "documentName": "$undefined",
        "documentType": "$undefined",
        "effFromBegin": "$undefined",
        "effFromEnd": "$undefined",
        "effStatus": "$undefined",
        "effToEnd": "$undefined",
        "effToFrom": "$undefined",
        "fieldTypeIds": "$undefined",
        "groupVbpl": True,
        "issueDateFrom": "$undefined",
        "issueDateTo": "$undefined",
        "keyword": "$undefined",
        "majorTypeIds": "$undefined",
        "matchMode": "all_words",
        "optionDoc": "all",
        "pageNumber": page_number,
        "pageSize": page_size,
        "sortBy": "issueDate",
        "sortDirection": "desc",
        "status": "$undefined",
        "useMb25": "$undefined",
    }
    return json.dumps([payload])   #? Body là mảng chứa một object.

#@ Đọc dòng dữ liệu "1:" trong phản hồi Flight và trả về JSON đã giải mã.
def parse_flight(text):
    #? Dữ liệu cần lấy nằm sau tiền tố "1:".
    for line in text.splitlines():
        if line.startswith("1:"):
            return json.loads(line[2:])
    return None

all_urls = []

resp = requests.post(URL, headers=HEADERS, data=build_body(1), timeout=15)
resp.encoding = "utf-8"          #? Giải mã UTF-8 để tránh lỗi hiển thị tiếng Việt.
print(resp.status_code)

data = parse_flight(resp.text)
if data is None:
    print(resp.text[:500])       #? In đoạn đầu để kiểm tra khi thiếu dòng "1:".
else:
    print("total:", data["total"])
    print(data.keys())
    for it in data["items"]:
        print( it["docNum"],
            "|", it["effStatus"]["name"],
            "|", it["issueDate"],
            "|", it["effFrom"],
        )

200
total: 57453
dict_keys(['total', 'pageNumber', 'pageSize', 'items'])
69/2026/TT-BXD | Chưa có hiệu lực | 2026-11-01T00:00:00 | 2026-11-01T00:00:00
368/2026/NĐ-CP | Chưa có hiệu lực | 2026-09-25T00:00:00 | 2026-10-01T00:00:00
138/2026/TT-BTC | Chưa có hiệu lực | 2026-09-25T00:00:00 | 2026-10-01T00:00:00
137/2026/TT-BTC | Chưa có hiệu lực | 2026-09-24T00:00:00 | 2029-01-01T00:00:00
47/2026/TT-NHNN | Chưa có hiệu lực | 2026-09-24T00:00:00 | 2027-07-01T00:00:00
362/2026/NĐ-CP | Chưa có hiệu lực | 2026-09-18T00:00:00 | 2026-11-02T00:00:00
06/2026/TT-BDTTG | Chưa có hiệu lực | 2026-09-18T00:00:00 | 2026-11-03T00:00:00
07/2026/TT-BDTTG | Chưa có hiệu lực | 2026-09-18T00:00:00 | 2026-11-03T00:00:00
359/2026/NĐ-CP | Chưa có hiệu lực | 2026-09-17T00:00:00 | 2026-11-06T00:00:00
70/2026/TT-BXD | Còn hiệu lực | 2026-09-17T00:00:00 | 2026-09-17T00:00:00


In [ ]:
#@ Tạo header và body cho request lấy nội dung từ URL văn bản.
def build_header_body(url):
    uuid = url.split("/")[-1]
    next_router_state_tree = "%5B%22%22%2C%7B%22children%22%3A%5B%5B%22locale%22%2C%22vi%22%2C%22d%22%5D%2C%7B%22children%22%3A%5B%22van-ban%22%2C%7B%22children%22%3A%5B%5B%22category%22%2C%22chi-tiet%22%2C%22d%22%5D%2C%7B%22children%22%3A%5B%5B%22id%22%2C%22" + uuid + "%22%2C%22d%22%5D%2C%7B%22children%22%3A%5B%22__PAGE__%22%2C%7B%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%2Ctrue%5D"
    headers = {
    "User-Agent": "Mozilla/5.0 (hoc-crawl-cho-mon-hoc)",
    "Content-Type": "text/plain;charset=UTF-8",
    "Next-Action": "0fb12b3561faa05adec51a82efb3e4f4f427f07b",
    "Accept": "text/x-component",
    "Accept-Language": "vi,en;q=0.9",
    "Origin": "https://vbpl.vn",
    "Referer": url,
    "Next-Router-State-Tree": next_router_state_tree,
}
    body = json.dumps([uuid])
    return headers, body
import re

#@ Tách HTML theo chunk ID từ phản hồi Flight; trả về None nếu không thấy.
def extract_html_chunk(content: bytes, chunk_id: str = "2") -> str | None:
    """Tách mảnh HTML dạng 'T' (text chunk) khỏi response flight.

    Định dạng giả định (suy từ quan sát): <id>:T<độ dài hex>,<nội dung>
    Độ dài đếm bằng BYTE, nên phải cắt trên bytes rồi mới decode.
    """
    #? Neo đầu dòng để tránh khớp nhầm chuỗi giống marker trong nội dung.
    pattern = re.compile(rb"(?:^|\n)" + chunk_id.encode() + rb":T([0-9a-fA-F]+),")
    m = pattern.search(content)
    if m is None:
        return None

    length = int(m.group(1), 16)          #? Độ dài nội dung tính bằng byte.
    start = m.end()                        #? HTML bắt đầu ngay sau dấu phẩy.
    raw = content[start:start + length]

    if len(raw) < length:                  #! Phản hồi bị cụt so với độ dài khai báo.
        print(f"CẢNH BÁO: khai báo {length} byte nhưng chỉ có {len(raw)}")
    return raw.decode("utf-8")

from bs4 import BeautifulSoup


#@ Lấy text, chuẩn hóa khoảng trắng và nối các dòng.
def clean_text(node, separator=" "):
    lines = [" ".join(line.split()) for line in node.get_text().splitlines()]
    return separator.join(line for line in lines if line)


#@ Chuyển các đoạn văn và bảng trong HTML thành chuỗi Markdown.
def html_to_md(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    content = soup.select_one("article.docx-page") or soup.body or soup

    for tag in content.find_all(["script", "style"]):
        tag.decompose()

    #? Giữ ranh giới dòng và đoạn văn, đặc biệt khi nằm trong ô bảng.
    for br in content.find_all("br"):
        br.replace_with("\n")
    for paragraph in content.find_all("p"):
        paragraph.append("\n")

    blocks = []

    for tag in content.find_all(["p", "table"]):
        #? Đoạn văn trong bảng sẽ được xử lý cùng bảng, tránh lấy trùng.
        if tag.find_parent("table") is not None:
            continue

        if tag.name == "p":
            text = clean_text(tag)
            if text:
                #? Đánh dấu bắt đầu Điều trước tiêu đề.
                if "prov-article" in tag.get("class", []):
                    blocks.append("PROV-ARTICLE")

                blocks.append(text)
            continue

        rows = []

        for tr in tag.find_all("tr"):
            row = []

            for cell in tr.find_all(["td", "th"], recursive=False):
                text = clean_text(cell, separator="<br>")
                text = text.replace("|", r"\|")  #? Tránh nhầm text với dấu chia cột.
                row.append(text)

                colspan = int(cell.get("colspan", 1))
                row.extend([""] * (colspan - 1))

            if row:
                rows.append(row)

        if not rows:
            continue

        width = max(len(row) for row in rows)

        #? Dùng header trống vì hàng đầu của bảng có thể không phải tiêu đề cột.
        table_lines = [
            "| " + " | ".join([""] * width) + " |",
            "| " + " | ".join(["---"] * width) + " |",
        ]

        for row in rows:
            row.extend([""] * (width - len(row)))
            table_lines.append("| " + " | ".join(row) + " |")

        if "prov-table" in tag.get("class", []):
            blocks.append("PROV_TABLE")

        blocks.append("\n".join(table_lines))

    return "\n\n".join(blocks)



In [ ]:
for doc_num, status, url in all_urls:
    file_name = "-".join(doc_num.split("/"))


    header, body = build_header_body(url)
    resp = requests.post(url, headers=header, data=body, timeout=15)
    html = extract_html_chunk(resp.content)

    if html is None:
        print("Không thấy mảnh '2:T...'. In 300 byte đầu để xem:")
        print(resp.content[:300])
    else:
        print("độ dài (ký tự):", len(html))
        print("kết thúc bằng </html>?", html.rstrip().endswith("</html>"))

        with open(f"{file_name}.html", "w", encoding="utf-8") as f:
            f.write(html)
        from pathlib import Path

        html_path = Path(f"{file_name}.html")
        html = html_path.read_text(encoding="utf-8")

        markdown = html_to_md(html)
        html_path.with_suffix(".md").write_text(markdown, encoding="utf-8")
    print()

độ dài (ký tự): 501363
kết thúc bằng </html>? True

độ dài (ký tự): 286612
kết thúc bằng </html>? True

độ dài (ký tự): 181077
kết thúc bằng </html>? True

độ dài (ký tự): 20629
kết thúc bằng </html>? True

độ dài (ký tự): 467701
kết thúc bằng </html>? True

độ dài (ký tự): 48020
kết thúc bằng </html>? True

độ dài (ký tự): 32735
kết thúc bằng </html>? True

độ dài (ký tự): 61911
kết thúc bằng </html>? True

độ dài (ký tự): 93332
kết thúc bằng </html>? True

độ dài (ký tự): 29394
kết thúc bằng </html>? True

